In [1]:
from datetime import datetime, timedelta
import pandas as pd
import requests


def fetch_exchange_rates_dataset():
  # Define base currency (IDR) and target major global currencies
  base_currency = "IDR"
  target_currencies = ["USD", "EUR", "SGD", "JPY", "GBP", "AUD", "CNY", "MYR"]

  # Define date range for the past 30 days up to yesterday (Exchange rate history)
  end_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
  start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")

  all_records = []
  print(
      f"Fetching historical exchange rate data from {start_date} to"
      f" {end_date}..."
  )

  # Loop through each target currency to fetch historical rates against IDR
  for currency in target_currencies:
    # Public exchange rate API endpoint (Frankfurter API)
    url = f"https://api.frankfurter.app/{start_date}..{end_date}"
    params = {"from": currency, "to": base_currency}

    response = requests.get(url, params=params)

    if response.status_code == 200:
      data = response.json()
      rates_history = data.get("rates", {})

      # Iterate through each date and its corresponding rate value
      for date_str, rate_dict in rates_history.items():
        exchange_rate = rate_dict.get(base_currency)

        if exchange_rate:
          all_records.append({
              "date": date_str,
              "base_currency": currency,
              "target_currency": base_currency,
              "exchange_rate": exchange_rate,
          })
    else:
      print(f"Failed to fetch exchange rate data for {currency}")

  # Convert collected records into a Pandas DataFrame and export as a CSV file
  if all_records:
    df = pd.DataFrame(all_records)

    # Sort data chronologically by date and currency symbol
    df = df.sort_values(by=["date", "base_currency"]).reset_index(drop=True)

    filename = "indonesia_exchange_rates_timeseries.csv"
    df.to_csv(filename, index=False)

    print(
        f"\nSuccess! Total records collected: {len(df)} rows.\nDataset saved"
        f" as: '{filename}'"
    )

    # Automatically trigger file download if running inside Google Colab
    try:
      from google.colab import files

      files.download(filename)
    except ImportError:
      pass
  else:
    print("No exchange rate data collected.")


if __name__ == "__main__":
  fetch_exchange_rates_dataset()

Fetching historical exchange rate data from 2026-09-15 to 2026-10-14...

Success! Total records collected: 176 rows.
Dataset saved as: 'indonesia_exchange_rates_timeseries.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>